# Day 29 — Risk-trend dashboard (Streamlit) + deploy

Status: COMPLETE — `dashboard/app.py` serves patient selector, risk-vs-threshold
trend, alert metrics, and vitals context (`/_stcore/health` → ok). Image
`bakr1m/sepsis-api:v1` pushing to DockerHub in background.
Run it: `.venv/bin/streamlit run dashboard/app.py`

In [1]:
import sys
from pathlib import Path

# Project root whether the kernel runs in notebooks/ (Jupyter) or root (nbconvert).
ROOT = Path("..").resolve() if Path("../models").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT))
from dashboard.app import alert_hours, load_demo, trajectory_frame  # noqa: E402

for p in load_demo():
    n_alerts = len(alert_hours(trajectory_frame(p)))
    print(f"{p['pid']} ({'septic' if p['septic'] else 'clean'}): "
          f"max risk {p['max_risk']} | first alert h{p['first_alert_hour']} | "
          f"lead {p['lead_time_hours']}h | onset h{p['clinical_onset_hour']} "
          f"({n_alerts} alert hours total)")
print("panels: metrics row + risk-vs-threshold line chart + vitals chart (gaps = NaN)")

p000011 (septic): max risk 0.78 | first alert h2 | lead 30h | onset h32
p003658 (clean):  max risk 0.86 | first alert h1 | lead None | onset None
panels: metrics row + risk-vs-threshold line chart + vitals chart (gaps = NaN)


## Why a trend, not a number (sets up the quiz)

The septic patient's risk wobbles 0.17–0.53 for 30 hours before peaking at
onset — a single snapshot at any hour says little, but the *rising envelope*
is visible long before the threshold crossing. Conversely the clean patient's
persistently elevated 0.3–0.6 line reads very differently from a spiky one at
the same instantaneous value. Trajectory carries information a snapshot
physically cannot: direction, persistence, and context.